# AI 06 - Ev tapşırığı
# Data Preprocessing və Regression Metrics - konspekt

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. One-Hot Encoding vs Factorize

Model mətn (kateqorik) dəyərlərlə işləyə bilmir, ona görə onları rəqəmə çevirmək lazımdır.

## One-Hot Encoding
Hər kateqoriya üçün ayrıca sütun yaradılır. Sətirdə həmin kateqoriya varsa **1**, yoxdursa **0** yazılır.

| color | → | color_blue | color_green | color_red |
|---|---|---|---|---|
| red | | 0 | 0 | 1 |
| blue | | 1 | 0 | 0 |

## pd.factorize()
Hər unikal dəyərə bir rəqəm (0, 1, 2 ...) verir. Bir sütun qalır.

## Fərqlər
- OHE çoxlu sütun yaradır, factorize bir sütun saxlayır.
- Factorize-da rəqəmlər arasında "sıra" yaranır (2 > 1 > 0), amma əslində red > blue deyil. Xətti modellər bunu səhv başa düşə bilər.
- Kateqoriya çox olanda OHE çoxlu sütun yaradır (yaddaş, sürət problemi).

## Nə vaxt hansı?
- **OHE** - kateqoriyalar arasında sıra yoxdursa və sayı azdırsa, xüsusən xətti modellərdə.
- **Factorize** - ağac əsaslı modellərdə (Decision Tree, Random Forest), sıralı (ordinal) datalarda və ya kateqoriya sayı çox olanda.

In [ ]:
df = pd.DataFrame({'color': ['red', 'blue', 'green', 'blue', 'red']})
df

In [ ]:
codes, uniques = pd.factorize(df['color'])
print(codes)
print(uniques)

In [ ]:
pd.get_dummies(df, columns=['color'])

In [ ]:
ohe = OneHotEncoder()
encoded = ohe.fit_transform(df[['color']]).toarray()
pd.DataFrame(encoded, columns=ohe.get_feature_names_out())

# 2. SimpleImputer

Datalarda çatışmayan (NaN) dəyərləri doldurmaq üçün istifadə olunur. Çünki əksər modellər NaN ilə işləmir.

Strategiyalar:
- `mean` - sütunun orta qiyməti ilə doldurur (yalnız rəqəmlər)
- `median` - median ilə doldurur, outlier olanda mean-dən yaxşıdır
- `most_frequent` - ən çox təkrarlanan dəyər, kateqorik sütunlar üçün də olur
- `constant` - bizim verdiyimiz sabit dəyər (`fill_value`)

Vacib: imputer-i `fit` yalnız train datada edilir, sonra test-ə `transform` edilir.

In [ ]:
data = pd.DataFrame({
    'age': [25, np.nan, 30, 35, np.nan],
    'city': ['Baku', 'Ganja', np.nan, 'Baku', 'Baku']
})
data

In [ ]:
for strategy in ['mean', 'median']:
    imp = SimpleImputer(strategy=strategy)
    print(strategy, imp.fit_transform(data[['age']]).ravel())

In [ ]:
imp_freq = SimpleImputer(strategy='most_frequent')
print(imp_freq.fit_transform(data[['city']]).ravel())

imp_const = SimpleImputer(strategy='constant', fill_value='Unknown')
print(imp_const.fit_transform(data[['city']]).ravel())

# 3. Regression Metrics

# MAE - Mean absolute error - Orta mütləq xəta
$$
MAE = \frac{1}{n}\sum_{i=1}^n|y_i - \hat{y_i}|
$$
Orta hesabla proqnoz nə qədər səhvdir. Vahidi hədəflə eynidir (məs. AZN). Outlier-lərə az həssasdır.

# MSE - Mean squared error - Orta kvadratik xəta
$$
MSE = \frac{1}{n}\sum_{i=1}^n(y_i - \hat{y_i})^2
$$
Xətalar kvadrata yüksəldilir, ona görə böyük səhvlər daha çox cəzalandırılır. Vahidi kvadratdır (AZN²), şərh etmək çətindir.

# RMSE - Root Mean squared error
$$
RMSE = \sqrt{MSE}
$$
MSE-nin kökü, vahidi yenə hədəflə eynidir. Böyük xətalara MAE-dən çox həssasdır. RMSE ≥ MAE olur.

# $R^2$ - Determinasiya əmsalı
$$
R^2 = 1 - \frac{\sum(y_i - \hat{y_i})^2}{\sum(y_i - \bar{y})^2}
$$
Model hədəfin dəyişkənliyinin neçə faizini izah edir. 1 - ideal, 0 - sadəcə orta qiymət qədər, mənfi - ortadan da pis.

**Fərq:** MAE, MSE, RMSE - xətadır (nə qədər kiçik o qədər yaxşı), R² isə keyfiyyət göstəricisidir (1-ə yaxın yaxşıdır).

In [ ]:
y_true = np.array([100, 150, 200, 250, 300])
y_pred = np.array([110, 140, 210, 240, 330])

In [ ]:
# eli ile hesablama
mae_manual = np.mean(np.abs(y_true - y_pred))
mse_manual = np.mean((y_true - y_pred)**2)
rmse_manual = np.sqrt(mse_manual)
r2_manual = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)
print(mae_manual, mse_manual, rmse_manual, r2_manual)

In [ ]:
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f'''
MAE: {mae:.2f}
MSE: {mse:.2f}
RMSE: {rmse:.2f}
R2: {r2:.2f}
''')